# Data inventory and loader

This notebook reads every file in `data/`. The CSV is loaded directly into pandas. The five `.fst` files are R/FST files, so their previews are read through R's `fst` package without loading all several gigabytes into the Python kernel at once.

Install the prerequisites once if needed:

```bash
python -m pip install pandas
R -e 'install.packages("fst", repos="https://cloud.r-project.org")'
```

In [1]:
from pathlib import Path
import csv
import shutil
import subprocess
import tempfile

import pandas as pd

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

files = sorted(p for p in DATA_DIR.iterdir() if p.is_file())
inventory = pd.DataFrame({
    "file": [p.name for p in files],
    "format": [p.suffix.lstrip(".").lower() for p in files],
    "size_gb": [round(p.stat().st_size / 1024**3, 3) for p in files],
})
inventory

,file,format,size_gb
0,.DS_Store,,0.000
1,analysis_dataset_thresh_90_1st_day_included_ho...,fst,0.556
2,analysis_dataset_thresh_95_1st_day_included_ho...,fst,0.340
3,analysis_dataset_thresh_97_1st_day_included_ho...,fst,0.244
4,analysis_dataset_thresh_99_1st_day_included_ho...,fst,0.074
5,gva-data-2014-2023.csv,csv,0.058
6,gva_satellite_sae_2014_2022_merged.csv,csv,0.168
7,satellite_embeddings_2014_2022.csv,csv,0.048
8,satellite_sae_features_2014_2022.csv,csv,0.052
9,spectral_indices_2014_2022.csv,csv,0.001


In [2]:
csv_path = DATA_DIR / "gva-data-2014-2023.csv"
gva = pd.read_csv(csv_path, encoding="utf-8-sig")
gva["Date"] = pd.to_datetime(gva["Date"], format="%d-%b-%y")
print(f"gva: {gva.shape[0]:,} rows x {gva.shape[1]:,} columns")
display(gva.head())

gva: 389,842 rows x 15 columns


,Incident ID,Date,State,City,County,Business or School,Address,Latitude,Longitude,# Victims Killed,# Victims Injured,# Suspect Killed,# Suspects Injured,# Suspects Arrested,Incident Characteristics
0,2792286,2023-12-31,Ohio,Lima,NaN,NaN,2226 N Glenwood Ave,40.7672,-84.1367,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)"
1,2791280,2023-12-31,Tennessee,Memphis,NaN,Edgewater Apartments,5800 block of Edgewater Cove,35.1800,-89.8677,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)\n..."
2,2794085,2023-12-31,Alaska,Anchorage,NaN,Point Woronzof Overlook,7251 Point Woronzof Rd,61.2020,-150.0210,0,1,0,0,0,Shot - Wounded/Injured\nDomestic Violence
3,2793872,2023-12-31,Florida,Quincy,NaN,Mane Attraction,110 E Washington St,30.5895,-84.5751,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)"
4,2790816,2023-12-31,New Mexico,Albuquerque,NaN,Valle de Atrisco Apartment Complex,9901 Ceja Vis Rd SW,35.0221,-106.7430,1,2,0,0,0,"Shot - Wounded/Injured\nShot - Dead (murder, a..."


In [3]:
def read_fst_preview(path: Path, nrows: int = 5) -> pd.DataFrame:
    """Read only the first nrows of an FST file through R."""
    rscript = shutil.which("Rscript")
    if rscript is None:
        raise RuntimeError("Rscript is required to read .fst files.")
    check = subprocess.run(
        [rscript, "--vanilla", "-e",
         "if (!requireNamespace('fst', quietly=TRUE)) quit(status=2)"],
        capture_output=True, text=True, check=False,
    )
    if check.returncode != 0:
        raise RuntimeError("Install the R package 'fst' before reading .fst files.")
    r_code = (
        'args <- commandArgs(trailingOnly=TRUE); '
        'x <- fst::read_fst(args[1], from=1, to=as.integer(args[3])); '
        "write.csv(x, args[2], row.names=FALSE, na=\"\")"
    )
    with tempfile.TemporaryDirectory() as tmp:
        output = Path(tmp) / "preview.csv"
        subprocess.run(
            [rscript, "--vanilla", "-e", r_code, str(path), str(output), str(nrows)],
            check=True, capture_output=True, text=True,
        )
        return pd.read_csv(output)

fst_paths = sorted(DATA_DIR.glob("*.fst"))
fst_previews = {path.stem: read_fst_preview(path) for path in fst_paths}
for name, frame in fst_previews.items():
    print(f"{name}: {frame.shape[1]:,} columns; preview rows = {len(frame)}")
    display(frame.head(4))

analysis_dataset_thresh_90_1st_day_included_hotter_included_2000_2022: 44 columns; preview rows = 5


,year,zcta,population,pct_black,pct_hispanic,pct_higher_education_attained,pct_poverty,pct_housing_owner_occupied,housing_units,median_home_value,...,q_zy97,q_zy99,split_id,doy,ind0,match_id,set_role,paired_ind0,hotter_control,match_row_id
0,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,1,144,144,41622513,control,41622513,0,7736312
1,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,1,144,144,41622514,control,41622514,0,7736313
2,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,3,145,145,5170737,control,5170737,0,4543885
3,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,160,160,36336967,control,36336967,0,7279885


analysis_dataset_thresh_95_1st_day_included_hotter_included_2000_2022: 44 columns; preview rows = 5


,year,zcta,population,pct_black,pct_hispanic,pct_higher_education_attained,pct_poverty,pct_housing_owner_occupied,housing_units,median_home_value,...,q_zy97,q_zy99,split_id,doy,ind0,match_id,set_role,paired_ind0,hotter_control,match_row_id
0,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,4,175,175,31037878,control,31037878,0,3200338
1,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,1,176,176,36336985,control,36336985,0,3416244
2,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,3,181,181,181,treated,31037889,0,1
3,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,182,182,182,treated,31037885,0,2


analysis_dataset_thresh_97_1st_day_included_hotter_included_2000_2022: 44 columns; preview rows = 5


,year,zcta,population,pct_black,pct_hispanic,pct_higher_education_attained,pct_poverty,pct_housing_owner_occupied,housing_units,median_home_value,...,q_zy97,q_zy99,split_id,doy,ind0,match_id,set_role,paired_ind0,hotter_control,match_row_id
0,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,164,164,15526100,control,15526100,0,1338034
1,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,182,182,182,treated,5170775,0,1
2,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,183,183,183,treated,31037889,0,2
3,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,4,184,184,184,treated,5170775,0,3


analysis_dataset_thresh_99_1st_day_included_hotter_included_2000_2022: 44 columns; preview rows = 5


,year,zcta,population,pct_black,pct_hispanic,pct_higher_education_attained,pct_poverty,pct_housing_owner_occupied,housing_units,median_home_value,...,q_zy97,q_zy99,split_id,doy,ind0,match_id,set_role,paired_ind0,hotter_control,match_row_id
0,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,1,179,179,20696708,control,20696708,0,401389
1,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,1,179,179,36336987,control,36336987,0,498370
2,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,2,183,183,183,treated,5170774,0,1
3,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,31.534,34.396,4,184,184,184,treated,5170773,0,2


unmatched_analysis_dataset_2014_2022: 37 columns; preview rows = 5


,year,zcta,population,pct_black,pct_hispanic,pct_higher_education_attained,pct_poverty,pct_housing_owner_occupied,housing_units,median_home_value,...,hw2_99_1std,hw2_90,hw2_95,hw2_97,hw2_99,q_zy90,q_zy95,q_zy97,q_zy99,split_id
0,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,0,0,0,0,0,28.728,30.406,31.534,34.396,1
1,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,0,0,0,0,0,28.728,30.406,31.534,34.396,3
2,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,0,0,0,0,0,28.728,30.406,31.534,34.396,3
3,2014,1001,17141,0.01126,0.059098,0.20168,0.081092,0.676614,7372,212900,...,0,0,0,0,0,28.728,30.406,31.534,34.396,3


## Full FST reads

Use the same helper with a larger `nrows` only when a bounded subset is needed. Reading an entire `.fst` file into pandas requires temporary CSV storage and several gigabytes of memory/disk, so the default workflow intentionally keeps previews bounded.

In [4]:
# Print variable names for every loaded dataset.
print(f"gva-data-2014-2023.csv ({len(gva.columns)} variables):")
for variable in gva.columns:
    print(f"  - {variable}")

for name, frame in fst_previews.items():
    print(f"\n{name}.fst ({len(frame.columns)} variables):")
    for variable in frame.columns:
        print(f"  - {variable}")

gva-data-2014-2023.csv (15 variables):
  - Incident ID
  - Date
  - State
  - City
  - County
  - Business or School
  - Address
  - Latitude
  - Longitude
  - # Victims Killed
  - # Victims Injured
  - # Suspect Killed
  - # Suspects Injured
  - # Suspects Arrested
  - Incident Characteristics

analysis_dataset_thresh_90_1st_day_included_hotter_included_2000_2022.fst (44 variables):
  - year
  - zcta
  - population
  - pct_black
  - pct_hispanic
  - pct_higher_education_attained
  - pct_poverty
  - pct_housing_owner_occupied
  - housing_units
  - median_home_value
  - median_household_income
  - climate_type
  - date
  - n_killed
  - n_injured
  - n_events
  - rmin
  - tmmx
  - pr
  - heat_index
  - hw2_prox90
  - hw2_prox95
  - hw2_prox97
  - hw2_prox99
  - hw2_90_1std
  - hw2_95_1std
  - hw2_97_1std
  - hw2_99_1std
  - hw2_90
  - hw2_95
  - hw2_97
  - hw2_99
  - q_zy90
  - q_zy95
  - q_zy97
  - q_zy99
  - split_id
  - doy
  - ind0
  - match_id
  - set_role
  - paired_ind0
  - hotter

In [5]:
for name, frame in fst_previews.items():
    print(f"\n{name}.fst:")
    print(frame[["date", "year", "zcta"]].head())


analysis_dataset_thresh_90_1st_day_included_hotter_included_2000_2022.fst:
         date  year  zcta
0  2014-05-24  2014  1001
1  2014-05-24  2014  1001
2  2014-05-25  2014  1001
3  2014-06-09  2014  1001
4  2014-06-17  2014  1001

analysis_dataset_thresh_95_1st_day_included_hotter_included_2000_2022.fst:
         date  year  zcta
0  2014-06-24  2014  1001
1  2014-06-25  2014  1001
2  2014-06-30  2014  1001
3  2014-07-01  2014  1001
4  2014-07-02  2014  1001

analysis_dataset_thresh_97_1st_day_included_hotter_included_2000_2022.fst:
         date  year  zcta
0  2014-06-13  2014  1001
1  2014-07-01  2014  1001
2  2014-07-02  2014  1001
3  2014-07-03  2014  1001
4  2014-07-18  2014  1001

analysis_dataset_thresh_99_1st_day_included_hotter_included_2000_2022.fst:
         date  year  zcta
0  2014-06-28  2014  1001
1  2014-06-28  2014  1001
2  2014-07-02  2014  1001
3  2014-07-03  2014  1001
4  2014-07-18  2014  1001

unmatched_analysis_dataset_2014_2022.fst:
         date  year  zcta
0  

## Merge GVA incidents with the satellite SAE features (2014-2022)

`satellite_sae_features_2014_2022.csv` has one row per ZCTA per year (LA + NYC, 2014-2022) with 4,096 sparse-autoencoder atom activations derived from that year's satellite imagery. `gva-data-2014-2023.csv` has per-incident latitude/longitude but no ZCTA, so the two can't be joined on a shared column directly.

Instead:
1. Restrict GVA incidents to 2014-2022 (the years with SAE features; the GVA data itself runs through 2023).
2. For each year, spatially join that year's incidents against that year's own LA/NYC ZCTA boundaries (`data/shp/LA`, `data/shp/NYC`) — the same boundaries used to build that year's satellite features — assigning each incident a `zcta`.
3. Merge on `zcta` and `year` together, since both the ZCTA boundaries and the SAE features are year-specific.

Incidents outside the LA/NYC ZCTA boundaries for their year are dropped, since no satellite features exist for them.

In [6]:
import geopandas as gpd

sae_path = DATA_DIR / "satellite_sae_features_2014_2022.csv"
sae = pd.read_csv(sae_path)
atom_cols = [c for c in sae.columns if c.startswith("atom_")]
print(f"satellite_sae_features_2014_2022: {sae.shape[0]:,} zcta-years x {len(atom_cols):,} atom features")
print(sae.groupby(["year", "city"]).size().unstack())
display(sae[["city", "year", "zcta", "filename"]].head())

satellite_sae_features_2014_2022: 3,355 zcta-years x 4,096 atom features
city   LA  NYC
year          
2014  161  211
2015  161  211
2016  162  211
2017  162  211
2018  162  211
2019  162  211
2020  162  211
2021  162  211
2022  162  211


,city,year,zcta,filename
0,LA,2014,90001,90001.tif
1,LA,2014,90002,90002.tif
2,LA,2014,90003,90003.tif
3,LA,2014,90004,90004.tif
4,LA,2014,90005,90005.tif


In [7]:
# Assign every GVA incident a ZCTA by spatially joining its coordinates
# against that year's own LA/NYC ZCTA boundaries (the same boundaries used
# to build that year's SAE features).
SAE_YEARS = range(2014, 2023)

gva_with_zcta_parts = []
gva_unmatched_parts = []

for year in SAE_YEARS:

    la_path = DATA_DIR / "shp" / "LA" / f"LA_ZCTA_{year}.shp"
    nyc_path = DATA_DIR / "shp" / "NYC" / f"NYC_ZCTA_{year}.shp"

    if not (la_path.exists() and nyc_path.exists()):
        print(f"{year}: skipped, ZCTA boundaries not found")
        continue

    la_zctas = gpd.read_file(la_path)
    nyc_zctas = gpd.read_file(nyc_path)
    zcta_boundaries = pd.concat([la_zctas, nyc_zctas], ignore_index=True)
    zcta_boundaries["zcta"] = zcta_boundaries["ZCTA5CE"].astype(int)

    gva_year = gva[gva["Date"].dt.year == year].dropna(subset=["Latitude", "Longitude"]).copy()
    gva_year["year"] = year

    gva_points = gpd.GeoDataFrame(
        gva_year,
        geometry=gpd.points_from_xy(gva_year["Longitude"], gva_year["Latitude"]),
        crs="EPSG:4326",
    ).to_crs(zcta_boundaries.crs)

    joined = gpd.sjoin(
        gva_points, zcta_boundaries[["zcta", "geometry"]], predicate="within", how="inner"
    ).drop(columns=["geometry", "index_right"])

    unmatched = gva_points.loc[~gva_points.index.isin(joined.index)].drop(columns=["geometry"])

    gva_with_zcta_parts.append(joined)
    gva_unmatched_parts.append(unmatched)

    print(
        f"{year}: {len(gva_year):,} incidents -> "
        f"{len(joined):,} inside an LA/NYC ZCTA, {len(unmatched):,} unmatched"
    )

gva_with_zcta = pd.concat(gva_with_zcta_parts, ignore_index=True)
gva_unmatched = pd.concat(gva_unmatched_parts, ignore_index=True)

print(f"\ntotal matched (inside an LA/NYC ZCTA), 2014-2022: {len(gva_with_zcta):,}")
print(f"total unmatched (outside LA/NYC, or no satellite ZCTA boundary), 2014-2022: {len(gva_unmatched):,}")

2014: 27,084 incidents -> 904 inside an LA/NYC ZCTA, 26,180 unmatched
2015: 32,468 incidents -> 1,102 inside an LA/NYC ZCTA, 31,366 unmatched


2016: 36,636 incidents -> 941 inside an LA/NYC ZCTA, 35,695 unmatched
2017: 37,508 incidents -> 913 inside an LA/NYC ZCTA, 36,595 unmatched


2018: 34,564 incidents -> 757 inside an LA/NYC ZCTA, 33,807 unmatched
2019: 36,290 incidents -> 976 inside an LA/NYC ZCTA, 35,314 unmatched


2020: 46,690 incidents -> 1,756 inside an LA/NYC ZCTA, 44,934 unmatched
2021: 48,471 incidents -> 1,773 inside an LA/NYC ZCTA, 46,698 unmatched


2022: 46,506 incidents -> 1,744 inside an LA/NYC ZCTA, 44,762 unmatched

total matched (inside an LA/NYC ZCTA), 2014-2022: 10,866
total unmatched (outside LA/NYC, or no satellite ZCTA boundary), 2014-2022: 335,351


In [8]:
# Unmatched incidents are mostly outside LA/NYC entirely (GVA is a national
# dataset); a smaller number are inside LA/NYC's metro area but outside the
# ZCTA boundaries used for the satellite imagery (e.g. bordering suburbs).
print("Unmatched incidents by year:")
print(gva_unmatched.groupby("year").size())

print("\nTop 15 states among unmatched incidents:")
print(gva_unmatched["State"].value_counts().head(15))

print("\nTop 15 cities among unmatched incidents:")
print(gva_unmatched["City"].value_counts().head(15))

display(gva_unmatched.head())

Unmatched incidents by year:
year
2014    26180
2015    31366
2016    35695
2017    36595
2018    33807
2019    35314
2020    44934
2021    46698
2022    44762
dtype: int64

Top 15 states among unmatched incidents:
State
Illinois          31337
Texas             23623
California        20008
Florida           18144
Pennsylvania      18093
Ohio              15644
North Carolina    12598
Louisiana         12471
Georgia           12204
Tennessee         11607
Michigan          10633
Maryland          10428
Missouri          10235
Virginia           9723
Indiana            8972
Name: count, dtype: int64

Top 15 cities among unmatched incidents:
City
Chicago         23877
Philadelphia    10653
Baltimore        7082
Houston          6181
Milwaukee        5251
Detroit          4990
Saint Louis      4842
New Orleans      4589
Washington       4514
Memphis          4451
Columbus         4108
Indianapolis     3554
San Antonio      3123
Jacksonville     3076
Cleveland        3005
Name: count, dty

,Incident ID,Date,State,City,County,Business or School,Address,Latitude,Longitude,# Victims Killed,# Victims Injured,# Suspect Killed,# Suspects Injured,# Suspects Arrested,Incident Characteristics,year
0,272425,2014-12-31,Colorado,Westminster,NaN,NaN,7700 block of Zenobia Court,39.8366,-105.0520,0,1,0,0,0,Shot - Wounded/Injured,2014
1,271863,2014-12-31,Florida,Saint Petersburg,NaN,NaN,12000 Fourth St. N.,27.8775,-82.6389,1,1,0,0,0,"Shot - Wounded/Injured\nShot - Dead (murder, a...",2014
2,272059,2014-12-31,Virginia,Hampton,NaN,NaN,50 block of Phelps Cir,37.0394,-76.3127,0,1,0,0,0,Shot - Wounded/Injured\nDomestic Violence\nPos...,2014
3,276313,2014-12-31,Washington,Renton,NaN,NaN,10415 S.E. 174th St,47.4468,-122.2010,0,1,0,0,0,Shot - Wounded/Injured\nArmed robbery with inj...,2014
4,272623,2014-12-31,Georgia,Temple,NaN,NaN,Bryan Miller Rd,33.8400,-84.9602,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)\n...",2014


In [9]:
merged = gva_with_zcta.merge(sae, on=["zcta", "year"], how="inner", suffixes=("", "_sae"))
print(f"merged: {merged.shape[0]:,} rows x {merged.shape[1]:,} columns")
print(merged.groupby(["year", "city"]).size().unstack())
display(merged.drop(columns=atom_cols).head())

merged: 10,866 rows x 4,115 columns
city   LA   NYC
year           
2014  399   505
2015  325   777
2016  329   612
2017  392   521
2018  303   454
2019  326   650
2020  425  1331
2021  483  1290
2022  629  1115


,Incident ID,Date,State,City,County,Business or School,Address,Latitude,Longitude,# Victims Killed,# Victims Injured,# Suspect Killed,# Suspects Injured,# Suspects Arrested,Incident Characteristics,year,zcta,city,filename
0,273436,2014-12-31,California,Los Angeles,NaN,NaN,6100 block of Wilton Place,33.9843,-118.3150,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)",2014,90047,LA,90047.tif
1,271919,2014-12-31,California,Los Angeles,NaN,NaN,81st Street and Western Avenue,33.9655,-118.3090,1,2,0,0,0,"Shot - Wounded/Injured\nShot - Dead (murder, a...",2014,90047,LA,90047.tif
2,272325,2014-12-30,New York,Brooklyn,NaN,NaN,940 Gates Avenue,40.6889,-73.9263,0,1,0,0,0,Shot - Wounded/Injured,2014,11221,NYC,11221.tif
3,271475,2014-12-30,New York,Brooklyn,NaN,NaN,712 Chauncey Street,40.6850,-73.9074,1,0,0,0,0,"Shot - Dead (murder, accidental, or suicide)\n...",2014,11207,NYC,11207.tif
4,279441,2014-12-30,California,Los Angeles,NaN,NaN,1300 block of Reliance Street,34.2289,-118.4240,0,1,0,0,0,Shot - Wounded/Injured,2014,91331,LA,91331.tif


## Incidents dropped during the SAE merge

The spatial join above already drops incidents outside an LA/NYC ZCTA. A separate, smaller drop can happen here: an incident can be spatially inside a (zcta, year) that has no matching row in `satellite_sae_features_2014_2022.csv` — e.g. a ZCTA whose TIFF crop failed in `04_split_tiff.py` and so was skipped from feature extraction. Since `merged` uses an inner join on `["zcta", "year"]`, those incidents are silently dropped from `merged`; this section surfaces them explicitly.

In [10]:
# Anything in gva_with_zcta (spatially matched to a ZCTA) that didn't survive
# the inner join onto sae was dropped because that (zcta, year) has no SAE
# features row.
matched_incident_ids = set(merged["Incident ID"])
merge_unmatched = gva_with_zcta[~gva_with_zcta["Incident ID"].isin(matched_incident_ids)]

print(f"incidents matched to a ZCTA but dropped by the SAE merge: {len(merge_unmatched):,}")

if len(merge_unmatched):
    print("\nBy year:")
    print(merge_unmatched.groupby("year").size())

    print("\nBy (zcta, year) - ZCTAs with no SAE feature row:")
    print(
        merge_unmatched.groupby(["zcta", "year"]).size()
        .rename("n_incidents").reset_index()
        .sort_values("n_incidents", ascending=False)
    )

    display(merge_unmatched.head())

incidents matched to a ZCTA but dropped by the SAE merge: 0


In [11]:
out_path = DATA_DIR / "gva_satellite_sae_2014_2022_merged.csv"
merged.to_csv(out_path, index=False)
print(f"saved {len(merged):,} rows to {out_path}")

saved 10,866 rows to ../data/gva_satellite_sae_2014_2022_merged.csv
